# FABSA Preprocessing Pipeline
Steps: deduplication → lexical normalisation → demojize → noise removal → whitespace strip → DeBERTa-v3 pairwise formatting

In [1]:
import ast
import re
from html import unescape

import contractions   # pip install contractions
import emoji
import pandas as pd

## Constants

In [2]:
DATA_PATH   = "../data/FABSA_train.csv"
OUTPUT_PATH = "../data/FABSA_train_preprocessed.csv"

ASPECT_CATEGORIES = [
    "account-management.account-access",
    "company-brand.competitor",
    "company-brand.general-satisfaction",
    "company-brand.reviews",
    "logistics-rides.speed",
    "online-experience.app-website",
    "purchase-booking-experience.ease-of-use",
    "staff-support.attitude-of-staff",
    "staff-support.email",
    "staff-support.phone",
    "value.discounts-promotions",
    "value.price-value-for-money",
]

ASPECT_READABLE = {
    "account-management.account-access":        "account management: account access",
    "company-brand.competitor":                 "company brand: competitor",
    "company-brand.general-satisfaction":       "company brand: general satisfaction",
    "company-brand.reviews":                    "company brand: reviews",
    "logistics-rides.speed":                    "logistics: speed",
    "online-experience.app-website":            "online experience: app or website",
    "purchase-booking-experience.ease-of-use":  "purchase or booking experience: ease of use",
    "staff-support.attitude-of-staff":          "staff support: attitude of staff",
    "staff-support.email":                      "staff support: email",
    "staff-support.phone":                      "staff support: phone",
    "value.discounts-promotions":               "value: discounts and promotions",
    "value.price-value-for-money":              "value: price value for money",
}

SENTIMENT_MAP = {"1": "positive", "-1": "negative", "0": "neutral"}

## 1. Load

In [3]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded: {len(df):,} rows")
df.head(2)

Loaded: 7,930 rows


,id,org_index,data_source,industry,text,labels,label_codes
0,301972057,600,Trustpilot,Price Comparison,My experience is only around the Parking forum...,"[array(['Staff support: Attitude of staff', 'n...","['staff-support.attitude-of-staff.-1', 'compan..."
1,301982453,514,Google Play,Banking,"I love it so handy, plus I hate my bank so it ...","[array(['Company brand: General satisfaction',...","['company-brand.general-satisfaction.1', 'comp..."


## EDA: Conflicting Polarity on the Same (Text, Aspect)

In [4]:
import ast

def parse_label_codes_raw(raw):
    if not isinstance(raw, str):
        return []
    try:
        return ast.literal_eval(raw)
    except Exception:
        return []

# Explode each label code into its own row while still on the raw df
raw_df = pd.read_csv(DATA_PATH)
raw_df["codes"] = raw_df["label_codes"].apply(parse_label_codes_raw)

exploded = raw_df[["id", "text", "codes"]].explode("codes").dropna(subset=["codes"])

# Split aspect code from sentiment score (last segment)
exploded[["aspect", "sent_score"]] = (
    exploded["codes"].str.rsplit(".", n=1, expand=True)
)

# ── Case A: same (text, aspect) in multiple rows with different polarities ──
# (could survive dedup if text differs by whitespace etc.)
cross_row = (
    exploded.groupby(["text", "aspect"])["sent_score"]
    .nunique()
    .reset_index(name="n_polarities")
)
cross_conflicts = cross_row[cross_row["n_polarities"] > 1]
print(f"Case A – same text + aspect, >1 polarity across rows: {len(cross_conflicts):,}")
if len(cross_conflicts):
    sample_text = cross_conflicts.iloc[0]["text"]
    sample_aspect = cross_conflicts.iloc[0]["aspect"]
    display(exploded[
        (exploded["text"] == sample_text) & (exploded["aspect"] == sample_aspect)
    ][["id", "aspect", "sent_score"]])

# ── Case B: same id + aspect appears twice with different polarities ────────
within_row = (
    exploded.groupby(["id", "aspect"])["sent_score"]
    .nunique()
    .reset_index(name="n_polarities")
)
within_conflicts = within_row[within_row["n_polarities"] > 1]
print(f"Case B – same review id + aspect, >1 polarity within one row: {len(within_conflicts):,}")
if len(within_conflicts):
    display(within_conflicts.head(5))

print("\nSentiment score distribution across all labels:")
print(exploded["sent_score"].value_counts())

Case A – same text + aspect, >1 polarity across rows: 73


,id,aspect,sent_score
6834,301982985,online-experience.app-website,-1
6834,301982985,online-experience.app-website,1


Case B – same review id + aspect, >1 polarity within one row: 73


,id,aspect,n_polarities
281,301971638,staff-support.attitude-of-staff,2
1635,301972713,staff-support.attitude-of-staff,2
2569,301973629,staff-support.attitude-of-staff,2
2651,301973717,staff-support.attitude-of-staff,2
3080,301974176,company-brand.general-satisfaction,2



Sentiment score distribution across all labels:
sent_score
1     9122
-1    4366
0      510
Name: count, dtype: int64


## 2. Deduplication

In [5]:
before = len(df)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Removed {before - len(df):,} duplicate rows  →  {len(df):,} remaining")

Removed 312 duplicate rows  →  7,618 remaining


## 3 – 7. Text Cleaning Pipeline

In [6]:
LEXICAL_MAP = {
    "w/o":   "without",
    "w/":    "with",
    "b4":    "before",
    "plz":   "please",
    "pls":   "please",
    "thx":   "thanks",
    "idk":   "i don't know",
    "fyi":   "for your information",
    "approx":"approximately",
    "esp":   "especially",
    "msg":   "message",
    "amt":   "amount",
    "qty":   "quantity",
}

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # 3) Lexical normalisation via contractions library
    text = contractions.fix(text)

    # 3b) Informal abbreviation normalisation
    for informal, standard in LEXICAL_MAP.items():
        text = re.sub(rf"\b{re.escape(informal)}\b", standard, text, flags=re.IGNORECASE)

    # 4a) Demojize – emoji → spaced description token, before stripping non-ascii
    text = emoji.demojize(text, delimiters=(" ", " "))

    # 4b) Unescape HTML entities (&amp; &lt; etc.)
    text = unescape(text)

    # 4c) Mask URLs with [URL] – preserve signal that a URL existed
    text = re.sub(r"https?://\S+|www\.\S+", "[URL]", text)

    # 4d) Strip HTML / XML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # 4e) Strip email addresses
    text = re.sub(r"\S+@\S+\.\S+", "", text)

    # 4f) Keep alphanumeric + core punctuation + [URL] placeholder; collapse rest
    text = re.sub(r"[^a-z0-9\s.,!?;:'\"\-_\[\]]", " ", text)

    # 4g) Collapse repeated punctuation (e.g. "!!!" → "!")
    text = re.sub(r"([.,!?;:])\1+", r"\1", text)

    # 5) Whitespace strip
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["clean_text"] = df["text"].apply(clean_text)
print("Cleaning done. Sample:")
df[["text", "clean_text"]].head(3)

Cleaning done. Sample:


,text,clean_text
0,My experience is only around the Parking forum...,my experience is only around the parking forum...
1,"I love it so handy, plus I hate my bank so it ...","i love it so handy, plus i hate my bank so it ..."
2,Sometimes it takes,sometimes it takes


## 8. Label Parsing

In [7]:
def parse_label_codes(raw) -> list:
    """
    Parse label_codes like ['staff-support.attitude-of-staff.-1', ...]
    into [(aspect_code, sentiment_str), ...]
    """
    if not isinstance(raw, str):
        return []
    try:
        codes = ast.literal_eval(raw)
    except Exception:
        return []

    result = []
    for code in codes:
        # last segment is the sentiment score: 1, -1, or 0
        parts = code.rsplit(".", 1)
        if len(parts) != 2:
            continue
        aspect, sent_code = parts
        sentiment = SENTIMENT_MAP.get(sent_code)
        if sentiment and aspect in ASPECT_CATEGORIES:
            result.append((aspect, sentiment))
    return result


df["parsed_labels"] = df["label_codes"].apply(parse_label_codes)
df[["label_codes", "parsed_labels"]].head(3)

,label_codes,parsed_labels
0,"['staff-support.attitude-of-staff.-1', 'compan...","[(staff-support.attitude-of-staff, negative), ..."
1,"['company-brand.general-satisfaction.1', 'comp...","[(company-brand.general-satisfaction, positive..."
2,['company-brand.general-satisfaction.-1'],"[(company-brand.general-satisfaction, negative)]"


## 9. DeBERTa-v3 Pairwise Input Formatting

Model: `yangheng/deberta-v3-base-absa-v1.1` (PyABSA ACSC).

Per the model documentation (Yang et al., 2023), the review is the **primary input**
and the aspect is passed as `text_pair`:

```python
tokenizer(text=sentence_a, text_pair=sentence_b)
# → [CLS] review [SEP] aspect [SEP]
```

The tokenizer handles `[CLS]` / `[SEP]` insertion automatically (Devlin et al., 2019;
He et al., 2021). `formatted_input` makes the structure explicit for inspection only.

In [8]:
# yangheng/deberta-v3-base-absa-v1.1 (PyABSA ACSC) pairwise format:
#   tokenizer(text=review, text_pair=aspect)
# → [CLS] review [SEP] aspect [SEP]
#
# sentence_a (primary)  = review text
# sentence_b (text_pair) = aspect category description
#
# The tokenizer inserts [CLS] / [SEP] automatically.
# formatted_input is a human-readable inspection string only.

records = []

for _, row in df.iterrows():
    for aspect_code, sentiment in row["parsed_labels"]:
        aspect_text = ASPECT_READABLE[aspect_code]
        records.append({
            "id":              row["id"],
            "data_source":     row["data_source"],
            "industry":        row["industry"],
            "text":      row["clean_text"],   # review → primary input
            "aspect":      aspect_text,          # aspect → text_pair
            "aspect_code":     aspect_code,
            "sentiment":       sentiment,
            "formatted_input": f"[CLS] {row['clean_text']} [SEP] {aspect_text} [SEP]",
        })

pair_df = pd.DataFrame(records)
print(f"Expanded to {len(pair_df):,} (text, aspect) pairs from {len(df):,} reviews")
pair_df.head(4)

Expanded to 13,659 (text, aspect) pairs from 7,618 reviews


,id,data_source,industry,text,aspect,aspect_code,sentiment,formatted_input
0,301972057,Trustpilot,Price Comparison,my experience is only around the parking forum...,staff support: attitude of staff,staff-support.attitude-of-staff,negative,[CLS] my experience is only around the parking...
1,301972057,Trustpilot,Price Comparison,my experience is only around the parking forum...,company brand: reviews,company-brand.reviews,negative,[CLS] my experience is only around the parking...
2,301972057,Trustpilot,Price Comparison,my experience is only around the parking forum...,company brand: general satisfaction,company-brand.general-satisfaction,negative,[CLS] my experience is only around the parking...
3,301982453,Google Play,Banking,"i love it so handy, plus i hate my bank so it ...",company brand: general satisfaction,company-brand.general-satisfaction,positive,"[CLS] i love it so handy, plus i hate my bank ..."


In [9]:
before_conflict = len(pair_df)
conflict_mask = pair_df.groupby(["text", "aspect"])["sentiment"].transform("nunique") > 1
pair_df = pair_df[~conflict_mask].reset_index(drop=True)
print(f"Removed {before_conflict - len(pair_df):,} conflicting rows "
      f"({(before_conflict - len(pair_df)) // 2} pairs)  →  {len(pair_df):,} remaining")

Removed 146 conflicting rows (73 pairs)  →  13,513 remaining


## 10. Same-Aspect Conflict Removal (Section 5.4)

Some reviews express both positive and negative opinions about the same aspect in
different sentences. FABSA captures both, producing rows like:

```
("great app. crashes often", app-website, positive)
("great app. crashes often", app-website, negative)
```

These produce the **identical model input** `(review, aspect)` but demand contradictory
outputs — contradictory supervision. Both rows are removed following the proposal's
Section 5.4 recommendation: *"conflicting review-aspect pairs may be excluded for
standard three-class ACSC training"*.

Neither sentiment is preserved because there is no principled way to choose between them
without sentence-level annotations. The inference pipeline handles this at runtime via
span extraction.

## 10. Sanity Checks

In [10]:
print("Sentiment distribution:")
print(pair_df["sentiment"].value_counts())

print("\nAspect distribution:")
print(pair_df["aspect_code"].value_counts())

print("\nSample formatted input:")
print(pair_df["formatted_input"].iloc[0])

Sentiment distribution:
sentiment
positive    8725
negative    4283
neutral      505
Name: count, dtype: int64

Aspect distribution:
aspect_code
online-experience.app-website              3401
company-brand.general-satisfaction         2825
purchase-booking-experience.ease-of-use    2260
staff-support.attitude-of-staff            1057
value.price-value-for-money                1014
logistics-rides.speed                       995
company-brand.competitor                    610
account-management.account-access           472
value.discounts-promotions                  402
staff-support.phone                         181
company-brand.reviews                       176
staff-support.email                         120
Name: count, dtype: int64

Sample formatted input:
[CLS] my experience is only around the parking forum, so my review is based on this specific experience. as someone who needed information on pursuing actions around an unfair parking fine it was pretty good to read, although th

## 11. Save

In [11]:
pair_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(pair_df):,} rows  →  {OUTPUT_PATH}")

Saved 13,513 rows  →  ../data/FABSA_train_preprocessed.csv


---
## Validation Set Preprocessing
Load the official `validation` split from `jordiclive/FABSA` and apply
the identical pipeline (dedup → clean → parse → pairwise expand).
Saves to `../data/FABSA_val_preprocessed.csv` for use in `train.py`.

In [12]:
from datasets import load_dataset

VAL_OUTPUT_PATH = "../data/FABSA_val_preprocessed.csv"

ds_val = load_dataset("jordiclive/FABSA", split="validation")
df_val = ds_val.to_pandas()
print(f"Validation rows loaded : {len(df_val):,}")
df_val.head(3)

Validation rows loaded : 1,057


,id,org_index,data_source,industry,text,labels,label_codes
0,610309432,5827,Google Play,Consulting,How do I retrieve accounts that people have ve...,"[[Account management: Account access, neutral]]",['account-management.account-access.0']
1,301974039,616,Trustpilot,Fashion,"Super fast delivery, even with the situation w...","[[Logistics rides: Speed, positive]]",['logistics-rides.speed.1']
2,301983653,549,Google Play,Travel Booking,Great tool for choising of places to visit,"[[Online experience: App website, positive], [...","['online-experience.app-website.1', 'company-b..."


In [13]:
# Deduplication
before_val = len(df_val)
df_val = df_val.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Removed {before_val - len(df_val):,} duplicates → {len(df_val):,} remaining")

# Apply the same clean_text and parse_label_codes defined above
df_val["clean_text"]    = df_val["text"].apply(clean_text)
df_val["parsed_labels"] = df_val["label_codes"].apply(parse_label_codes)
print("Cleaning and label parsing done.")

Removed 15 duplicates → 1,042 remaining
Cleaning and label parsing done.


In [14]:
# Expand to (text, aspect) pairs — same format as training set
val_records = []
for _, row in df_val.iterrows():
    for aspect_code, sentiment in row["parsed_labels"]:
        aspect_text = ASPECT_READABLE[aspect_code]
        val_records.append({
            "id":              row["id"],
            "data_source":     row["data_source"],
            "industry":        row["industry"],
            "text":            row["clean_text"],
            "aspect":          aspect_text,
            "aspect_code":     aspect_code,
            "sentiment":       sentiment,
            "formatted_input": f"[CLS] {row['clean_text']} [SEP] {aspect_text} [SEP]",
        })

val_pair_df = pd.DataFrame(val_records)
print(f"Expanded to {len(val_pair_df):,} (text, aspect) pairs from {len(df_val):,} reviews")
val_pair_df.head(3)

Expanded to 1,843 (text, aspect) pairs from 1,042 reviews


,id,data_source,industry,text,aspect,aspect_code,sentiment,formatted_input
0,610309432,Google Play,Consulting,how do i retrieve accounts that people have ve...,account management: account access,account-management.account-access,neutral,[CLS] how do i retrieve accounts that people h...
1,301974039,Trustpilot,Fashion,"super fast delivery, even with the situation w...",logistics: speed,logistics-rides.speed,positive,"[CLS] super fast delivery, even with the situa..."
2,301983653,Google Play,Travel Booking,great tool for choising of places to visit,online experience: app or website,online-experience.app-website,positive,[CLS] great tool for choising of places to vis...


In [15]:
before_conflict = len(val_pair_df)
conflict_mask = val_pair_df.groupby(["text", "aspect"])["sentiment"].transform("nunique") > 1
val_pair_df = val_pair_df[~conflict_mask].reset_index(drop=True)
print(f"[val] Removed {before_conflict - len(val_pair_df):,} conflicting rows "
      f"({(before_conflict - len(val_pair_df)) // 2} pairs)  →  {len(val_pair_df):,} remaining")

[val] Removed 24 conflicting rows (12 pairs)  →  1,819 remaining


In [16]:
print("Sentiment distribution (validation):")
print(val_pair_df["sentiment"].value_counts())

print("\nAspect distribution (validation):")
print(val_pair_df["aspect_code"].value_counts())

Sentiment distribution (validation):
sentiment
positive    1199
negative     555
neutral       65
Name: count, dtype: int64

Aspect distribution (validation):
aspect_code
online-experience.app-website              453
company-brand.general-satisfaction         400
purchase-booking-experience.ease-of-use    325
value.price-value-for-money                128
staff-support.attitude-of-staff            124
logistics-rides.speed                      118
company-brand.competitor                    86
account-management.account-access           81
value.discounts-promotions                  51
company-brand.reviews                       22
staff-support.phone                         18
staff-support.email                         13
Name: count, dtype: int64


In [17]:
val_pair_df.to_csv(VAL_OUTPUT_PATH, index=False)
print(f"Saved {len(val_pair_df):,} rows → {VAL_OUTPUT_PATH}")

Saved 1,819 rows → ../data/FABSA_val_preprocessed.csv


---
## Test Set Preprocessing
Load the official `test` split from `jordiclive/FABSA` and apply
the identical pipeline (dedup → clean → parse → pairwise expand).

**Why clean the test set?**  
The raw text contains the same noise as train/val (emojis, HTML entities,
contractions, mixed case, URLs, etc.).  The DeBERTa model was trained on
`clean_text` output, so test inputs must go through the same transformation
to avoid distribution shift.  `evaluate.py` applies `clean_text` on the fly;
this cell saves the preprocessed version for offline analysis and
`llm_judge_eval.py` reference.

Saves to `../data/FABSA_test_preprocessed.csv`.

In [18]:
from datasets import load_dataset

TEST_OUTPUT_PATH = "../data/FABSA_test_preprocessed.csv"

ds_test = load_dataset("jordiclive/FABSA", split="test")
df_test = ds_test.to_pandas()
print(f"Test rows loaded : {len(df_test):,}")
df_test.head(3)

Test rows loaded : 1,587


,id,org_index,data_source,industry,text,labels,label_codes
0,301982094,514,Google Play,Banking,Very useful and easy.,"[[Purchase booking experience: Ease of use, po...",['purchase-booking-experience.ease-of-use.1']
1,301981085,369,Google Play,Ride Hailing,easy to use.gud response from customer care se...,"[[Staff support: Attitude of staff, positive],...","['staff-support.attitude-of-staff.1', 'online-..."
2,301986508,685,Google Play,Trading,money 😁,"[[Company brand: General satisfaction, positive]]",['company-brand.general-satisfaction.1']


In [19]:
# Deduplication
before_test = len(df_test)
df_test = df_test.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Removed {before_test - len(df_test):,} duplicates → {len(df_test):,} remaining")

# Apply the same clean_text and parse_label_codes defined above
df_test["clean_text"]    = df_test["text"].apply(clean_text)
df_test["parsed_labels"] = df_test["label_codes"].apply(parse_label_codes)
print("Cleaning and label parsing done.")

Removed 26 duplicates → 1,561 remaining
Cleaning and label parsing done.


In [20]:
# Expand to (text, aspect) pairs — same format as train/val sets
test_records = []
for _, row in df_test.iterrows():
    for aspect_code, sentiment in row["parsed_labels"]:
        aspect_text = ASPECT_READABLE[aspect_code]
        test_records.append({
            "id":              row["id"],
            "data_source":     row["data_source"],
            "industry":        row["industry"],
            "text":            row["clean_text"],
            "aspect":          aspect_text,
            "aspect_code":     aspect_code,
            "sentiment":       sentiment,
            "formatted_input": f"[CLS] {row['clean_text']} [SEP] {aspect_text} [SEP]",
        })

test_pair_df = pd.DataFrame(test_records)
print(f"Expanded to {len(test_pair_df):,} (text, aspect) pairs from {len(df_test):,} reviews")
test_pair_df.head(3)

Expanded to 2,783 (text, aspect) pairs from 1,561 reviews


,id,data_source,industry,text,aspect,aspect_code,sentiment,formatted_input
0,301982094,Google Play,Banking,very useful and easy.,purchase or booking experience: ease of use,purchase-booking-experience.ease-of-use,positive,[CLS] very useful and easy. [SEP] purchase or ...
1,301981085,Google Play,Ride Hailing,easy to use.gud response from customer care se...,staff support: attitude of staff,staff-support.attitude-of-staff,positive,[CLS] easy to use.gud response from customer c...
2,301981085,Google Play,Ride Hailing,easy to use.gud response from customer care se...,online experience: app or website,online-experience.app-website,negative,[CLS] easy to use.gud response from customer c...


In [21]:
print("Sentiment distribution (test):")
print(test_pair_df["sentiment"].value_counts())

print("\nAspect distribution (test):")
print(test_pair_df["aspect_code"].value_counts())

Sentiment distribution (test):
sentiment
positive    1796
negative     896
neutral       91
Name: count, dtype: int64

Aspect distribution (test):
aspect_code
online-experience.app-website              735
company-brand.general-satisfaction         570
purchase-booking-experience.ease-of-use    496
value.price-value-for-money                206
staff-support.attitude-of-staff            201
logistics-rides.speed                      188
company-brand.competitor                   120
value.discounts-promotions                  89
account-management.account-access           79
staff-support.phone                         39
company-brand.reviews                       38
staff-support.email                         22
Name: count, dtype: int64


In [22]:
before_conflict = len(test_pair_df)
conflict_mask = test_pair_df.groupby(["text", "aspect"])["sentiment"].transform("nunique") > 1
test_pair_df = test_pair_df[~conflict_mask].reset_index(drop=True)
print(f"[test] Removed {before_conflict - len(test_pair_df):,} conflicting rows "
      f"({(before_conflict - len(test_pair_df)) // 2} pairs)  →  {len(test_pair_df):,} remaining")

[test] Removed 42 conflicting rows (21 pairs)  →  2,741 remaining


In [23]:
test_pair_df.to_csv(TEST_OUTPUT_PATH, index=False)
print(f"Saved {len(test_pair_df):,} rows → {TEST_OUTPUT_PATH}")

Saved 2,741 rows → ../data/FABSA_test_preprocessed.csv
